# Operaciones y Portfolio

A partir de la versión `1.12.0` la librería incorpora dos objetos que permiten agrupar patas y consultar su estado *en batch*: `Operation` y `Portfolio`. El objetivo es responder, para miles de operaciones a la vez, preguntas del tipo *¿cuál es el devengo, el capital vigente y el valor presente de cada pata a la fecha t?* o *¿qué flujos se liquidan entre t1 y t2?*, entregando el resultado en formato **columnar** (un `dict` de arrays de `numpy`), listo para construir un `pd.DataFrame` sin recorrer cashflow por cashflow.

Para ejecutar todos los ejemplos se debe importar la librería. Se sugiere utilizar siempre el alias `qcf`.

In [1]:
import qcfinancial as qcf

import aux_functions as aux
import pandas as pd
import numpy as np
import time

In [2]:
qcf.id()

'version: 1.12.0a3, build: 2026-07-19 00:30'

## `Operation`

Un objeto `Operation` es un contenedor **inmutable** de una o más patas (`Leg`), identificado por una clave numérica (`key`) que entrega el usuario. La librería no le atribuye ningún significado a esa clave: es simplemente el identificador con el que el resultado de las consultas vuelve al llamador. Las patas se numeran desde 1 en el orden de construcción (`leg_number`).

Se construye un calendario y dos patas a tasa fija que representarán un swap de monedas: se recibe CLP y se paga USD.

In [3]:
calendario = qcf.CalendarFactory.build(
    qcf.QCDate("2025-01-01"),
    10,
    [qcf.BusinessCalendarId.CLSA, ],
)

fecha_inicio = qcf.QCDate(15, 1, 2026)
fecha_final = qcf.QCDate(15, 1, 2029)

In [4]:
pata_clp = qcf.LegFactory.build_bullet_fixed_rate_leg(
    rec_pay=qcf.RecPay.RECEIVE,
    start_date=fecha_inicio,
    end_date=fecha_final,
    bus_adj_rule=qcf.BusyAdjRules.MODFOLLOW,
    settlement_periodicity=qcf.Tenor('6M'),
    settlement_stub_period=qcf.StubPeriod.NO,
    settlement_calendar=calendario,
    settlement_lag=0,
    initial_notional=100_000_000.0,
    amort_is_cashflow=True,
    interest_rate=qcf.QCInterestRate(.055, qcf.QCAct360(), qcf.QCLinearWf()),
    notional_currency=qcf.QCCLP(),
    is_bond=False,
)

pata_usd = qcf.LegFactory.build_bullet_fixed_rate_leg(
    rec_pay=qcf.RecPay.PAY,
    start_date=fecha_inicio,
    end_date=fecha_final,
    bus_adj_rule=qcf.BusyAdjRules.MODFOLLOW,
    settlement_periodicity=qcf.Tenor('6M'),
    settlement_stub_period=qcf.StubPeriod.NO,
    settlement_calendar=calendario,
    settlement_lag=0,
    initial_notional=100_000.0,
    amort_is_cashflow=True,
    interest_rate=qcf.QCInterestRate(.042, qcf.QCAct360(), qcf.QCLinearWf()),
    notional_currency=qcf.QCUSD(),
    is_bond=False,
)

Con ambas patas se da de alta la `Operation`. Se requiere sólo la clave y la lista de patas. La dirección *receive/pay* **no** se entrega ni se almacena: ya está expresada en los signos de los montos, que `LegFactory` fijó al construir cada pata a partir de su argumento `rec_pay`.

In [5]:
operacion = qcf.Operation(
    1,                       # key
    [pata_clp, pata_usd],    # legs
)

In [6]:
print(f'key: {operacion.get_key()}')
print(f'número de patas: {operacion.number_of_legs()}')

key: 1
número de patas: 2


La dirección de cada pata se lee del signo de sus flujos: positivos los que se reciben, negativos los que se pagan.

In [7]:
for i in range(operacion.number_of_legs()):
    monto = operacion.get_leg(i).get_cashflow_at(0).amount()
    print(f'pata {i + 1}: primer flujo {monto:,.2f} -> {"recibe" if monto > 0 else "paga"}')

pata 1: primer flujo 2,765,277.78 -> recibe
pata 2: primer flujo -2,111.67 -> paga


El método `get_leg` entrega la pata en el índice dado (base 0, de modo que `leg_number = índice + 1`).

In [8]:
df = aux.leg_as_dataframe(operacion.get_leg(0))
df.style.format(aux.format_dict)

,fecha_inicial,fecha_final,fecha_pago,nocional,amortizacion,interes,amort_es_flujo,flujo,moneda_nocional,valor_tasa,tipo_tasa
0,2026-01-15,2026-07-15,2026-07-15,"100,000,000.00",0.00,"2,765,277.78",True,"2,765,277.78",CLP,5.5000%,LinAct360
1,2026-07-15,2027-01-15,2027-01-15,"100,000,000.00",0.00,"2,811,111.11",True,"2,811,111.11",CLP,5.5000%,LinAct360
2,2027-01-15,2027-07-15,2027-07-15,"100,000,000.00",0.00,"2,765,277.78",True,"2,765,277.78",CLP,5.5000%,LinAct360
3,2027-07-15,2028-01-17,2028-01-17,"100,000,000.00",0.00,"2,841,666.67",True,"2,841,666.67",CLP,5.5000%,LinAct360
4,2028-01-17,2028-07-17,2028-07-17,"100,000,000.00",0.00,"2,780,555.56",True,"2,780,555.56",CLP,5.5000%,LinAct360
5,2028-07-17,2029-01-15,2029-01-15,"100,000,000.00","100,000,000.00","2,780,555.56",True,"102,780,555.56",CLP,5.5000%,LinAct360


## `Portfolio`

Un `Portfolio` es un contenedor de `Operation` indexado por su clave, que se construye una vez y se modifica incrementalmente con `add` y `remove`. Es el objeto que responde las consultas en batch.

Se agregan tres operaciones: la anterior y dos variantes con distinto plazo y nocional.

In [9]:
def build_operacion(key: int, nocional: float, meses: int) -> qcf.Operation:
    """
    Retorna una Operation de dos patas fijas (recibe CLP, paga USD).
    """
    fecha_termino = fecha_inicio.add_months(meses)
    clp = qcf.LegFactory.build_bullet_fixed_rate_leg(
        rec_pay=qcf.RecPay.RECEIVE,
        start_date=fecha_inicio,
        end_date=fecha_termino,
        bus_adj_rule=qcf.BusyAdjRules.MODFOLLOW,
        settlement_periodicity=qcf.Tenor('6M'),
        settlement_stub_period=qcf.StubPeriod.NO,
        settlement_calendar=calendario,
        settlement_lag=0,
        initial_notional=nocional,
        amort_is_cashflow=True,
        interest_rate=qcf.QCInterestRate(.055, qcf.QCAct360(), qcf.QCLinearWf()),
        notional_currency=qcf.QCCLP(),
        is_bond=False,
    )
    usd = qcf.LegFactory.build_bullet_fixed_rate_leg(
        rec_pay=qcf.RecPay.PAY,
        start_date=fecha_inicio,
        end_date=fecha_termino,
        bus_adj_rule=qcf.BusyAdjRules.MODFOLLOW,
        settlement_periodicity=qcf.Tenor('6M'),
        settlement_stub_period=qcf.StubPeriod.NO,
        settlement_calendar=calendario,
        settlement_lag=0,
        initial_notional=nocional / 1_000.0,
        amort_is_cashflow=True,
        interest_rate=qcf.QCInterestRate(.042, qcf.QCAct360(), qcf.QCLinearWf()),
        notional_currency=qcf.QCUSD(),
        is_bond=False,
    )
    return qcf.Operation(key, [clp, usd])

In [10]:
portfolio = qcf.Portfolio()
portfolio.add(operacion)
portfolio.add(build_operacion(2, 250_000_000.0, 24))
portfolio.add(build_operacion(3, 50_000_000.0, 60))

portfolio.size()

3

La clave debe ser única: agregar una operación con una clave ya presente levanta una excepción. Lo mismo ocurre al intentar eliminar una clave inexistente.

In [11]:
try:
    portfolio.add(build_operacion(3, 1_000.0, 12))
except ValueError as e:
    print(e)

Portfolio: operation with key 3 already exists.


## `states_at`: estado de cada pata a una fecha

El método `states_at` entrega, para cada pata viva de cada operación, su estado a la fecha `t`. El resultado es un `dict` de arrays de `numpy` **paralelos**: el elemento `i` de cada array se refiere a la misma pata, y las filas vienen ordenadas por `(op_key, leg_number)`.

Las columnas son:

- `op_key`: clave de la operación
- `leg_number`: número de la pata dentro de la operación (base 1)
- `currency`: **índice** dentro de `currency_legend`, no el código ISO
- `accrued_interest`: interés devengado a la fecha `t` del cupón vigente
- `outstanding_notional`: capital vigente del cupón vigente
- `interest_settling`, `amortization_settling`, `total_settling`: montos que liquidan **exactamente** en `t`
- `next_flow_date`: fecha del próximo flujo posterior a `t`, como serial de Excel (`0` si no quedan flujos)
- `present_value`: valor presente de los flujos posteriores a `t` (ver más abajo)

Además, la llave `currency_legend` contiene la lista de códigos ISO a la que apunta la columna `currency`.

In [12]:
estados = portfolio.states_at(qcf.QCDate(15, 4, 2026))
estados

{'op_key': array([1, 1, 2, 2, 3, 3]),
 'leg_number': array([1, 2, 1, 2, 1, 2], dtype=int32),
 'currency': array([0, 1, 0, 1, 0, 1], dtype=int32),
 'accrued_interest': array([ 1.3750e+06, -1.0500e+03,  3.4375e+06, -2.6250e+03,  6.8750e+05,
        -5.2500e+02]),
 'outstanding_notional': array([ 1.0e+08, -1.0e+05,  2.5e+08, -2.5e+05,  5.0e+07, -5.0e+04]),
 'interest_settling': array([0., 0., 0., 0., 0., 0.]),
 'amortization_settling': array([0., 0., 0., 0., 0., 0.]),
 'total_settling': array([0., 0., 0., 0., 0., 0.]),
 'next_flow_date': array([46218, 46218, 46218, 46218, 46218, 46218]),
 'present_value': array([nan, nan, nan, nan, nan, nan]),
 'currency_legend': ['CLP', 'USD']}

Para trabajar cómodamente se envuelve el resultado en un `pd.DataFrame`, resolviendo la moneda contra su leyenda y transformando los seriales de Excel en fechas.

In [13]:
def states_as_dataframe(estados: dict) -> pd.DataFrame:
    """
    Envuelve el resultado de Portfolio.states_at en un pd.DataFrame.
    """
    leyenda = estados['currency_legend']
    df = pd.DataFrame({k: v for k, v in estados.items() if k != 'currency_legend'})
    df['currency'] = [leyenda[i] for i in df['currency']]
    df['next_flow_date'] = [
        qcf.QCDate(int(s)).iso_code() if s > 0 else None for s in df['next_flow_date']
    ]
    return df

In [14]:
df = states_as_dataframe(estados)
df.style.format({
    'accrued_interest': '{0:,.2f}',
    'outstanding_notional': '{0:,.2f}',
    'interest_settling': '{0:,.2f}',
    'amortization_settling': '{0:,.2f}',
    'total_settling': '{0:,.2f}',
    'present_value': '{0:,.2f}',
})

,op_key,leg_number,currency,accrued_interest,outstanding_notional,interest_settling,amortization_settling,total_settling,next_flow_date,present_value
0,1,1,CLP,"1,375,000.00","100,000,000.00",0.00,0.00,0.00,2026-07-15,nan
1,1,2,USD,"-1,050.00","-100,000.00",0.00,0.00,0.00,2026-07-15,nan
2,2,1,CLP,"3,437,500.00","250,000,000.00",0.00,0.00,0.00,2026-07-15,nan
3,2,2,USD,"-2,625.00","-250,000.00",0.00,0.00,0.00,2026-07-15,nan
4,3,1,CLP,"687,500.00","50,000,000.00",0.00,0.00,0.00,2026-07-15,nan
5,3,2,USD,-525.00,"-50,000.00",0.00,0.00,0.00,2026-07-15,nan


Nótese que `present_value` viene con `NaN`: sin curvas de descuento no hay valorización posible. Nótese también que las columnas `*_settling` son cero, porque el 15-04-2026 no es fecha de liquidación de ninguna pata. Si se consulta justo en una fecha de pago, el flujo aparece en esas columnas y ya no forma parte del valor presente.

In [15]:
df = states_as_dataframe(portfolio.states_at(qcf.QCDate(15, 7, 2026)))
df.style.format({
    'accrued_interest': '{0:,.2f}',
    'outstanding_notional': '{0:,.2f}',
    'interest_settling': '{0:,.2f}',
    'amortization_settling': '{0:,.2f}',
    'total_settling': '{0:,.2f}',
    'present_value': '{0:,.2f}',
})

,op_key,leg_number,currency,accrued_interest,outstanding_notional,interest_settling,amortization_settling,total_settling,next_flow_date,present_value
0,1,1,CLP,0.00,"100,000,000.00","2,765,277.78",0.00,"2,765,277.78",2027-01-15,nan
1,1,2,USD,-0.00,"-100,000.00","-2,111.67",0.00,"-2,111.67",2027-01-15,nan
2,2,1,CLP,0.00,"250,000,000.00","6,913,194.44",0.00,"6,913,194.44",2027-01-15,nan
3,2,2,USD,-0.00,"-250,000.00","-5,279.17",0.00,"-5,279.17",2027-01-15,nan
4,3,1,CLP,0.00,"50,000,000.00","1,382,638.89",0.00,"1,382,638.89",2027-01-15,nan
5,3,2,USD,-0.00,"-50,000.00","-1,055.83",0.00,"-1,055.83",2027-01-15,nan


### Valor presente

Si se entrega el argumento `curves` —un `dict` que asocia código ISO de moneda con un objeto `ZeroCouponCurve`— el método calcula además el valor presente de cada pata, descontando los flujos posteriores a `t` con la curva de la moneda de la pata.

In [16]:
def zcc_plana(tasa: float) -> qcf.ZeroCouponCurve:
    """
    Retorna una ZeroCouponCurve plana en el valor `tasa`.
    """
    lvec = qcf.long_vec()
    vec = qcf.double_vec()
    for plazo in [1, 365, 3_650]:
        lvec.append(plazo)
        vec.append(tasa)
    curva = qcf.QCCurve(lvec, vec)
    lin = qcf.QCLinearInterpolator(curva)
    return qcf.ZeroCouponCurve(lin, qcf.QCInterestRate(.0, qcf.QCAct365(), qcf.QCCompoundWf()))

curvas = {
    'CLP': zcc_plana(.052),
    'USD': zcc_plana(.038),
}

In [17]:
df = states_as_dataframe(portfolio.states_at(qcf.QCDate(15, 4, 2026), curvas))
df.style.format({
    'accrued_interest': '{0:,.2f}',
    'outstanding_notional': '{0:,.2f}',
    'interest_settling': '{0:,.2f}',
    'amortization_settling': '{0:,.2f}',
    'total_settling': '{0:,.2f}',
    'present_value': '{0:,.2f}',
})

,op_key,leg_number,currency,accrued_interest,outstanding_notional,interest_settling,amortization_settling,total_settling,next_flow_date,present_value
0,1,1,CLP,"1,375,000.00","100,000,000.00",0.00,0.00,0.00,2026-07-15,"102,489,307.36"
1,1,2,USD,"-1,050.00","-100,000.00",0.00,0.00,0.00,2026-07-15,"-102,326.19"
2,2,1,CLP,"3,437,500.00","250,000,000.00",0.00,0.00,0.00,2026-07-15,"255,251,702.19"
3,2,2,USD,"-2,625.00","-250,000.00",0.00,0.00,0.00,2026-07-15,"-254,694.06"
4,3,1,CLP,"687,500.00","50,000,000.00",0.00,0.00,0.00,2026-07-15,"51,605,796.54"
5,3,2,USD,-525.00,"-50,000.00",0.00,0.00,0.00,2026-07-15,"-51,588.42"


Cuando falta la curva de la moneda de una pata, el valor presente de esa pata vuelve como `NaN` en lugar de levantar una excepción: las demás patas se valorizan igual. Se repite la consulta entregando sólo la curva CLP.

In [18]:
df = states_as_dataframe(portfolio.states_at(qcf.QCDate(15, 4, 2026), {'CLP': curvas['CLP']}))
df[['op_key', 'leg_number', 'currency', 'present_value']].style.format({'present_value': '{0:,.2f}'})

,op_key,leg_number,currency,present_value
0,1,1,CLP,"102,489,307.36"
1,1,2,USD,nan
2,2,1,CLP,"255,251,702.19"
3,2,2,USD,nan
4,3,1,CLP,"51,605,796.54"
5,3,2,USD,nan


## `flows_between`: flujos en un intervalo

El método `flows_between` entrega todos los flujos contractuales cuya fecha de liquidación cae en el intervalo **semiabierto** `(t1, t2]`: se excluye `t1` y se incluye `t2`. Esto permite recorrer el calendario en ventanas sucesivas sin contar dos veces el mismo flujo. Las filas vienen ordenadas por `(op_key, leg_number, settlement_date)`.

In [19]:
flujos = portfolio.flows_between(qcf.QCDate(31, 12, 2025), qcf.QCDate(31, 12, 2026))

def flows_as_dataframe(flujos: dict) -> pd.DataFrame:
    """
    Envuelve el resultado de Portfolio.flows_between en un pd.DataFrame.
    """
    leyenda = flujos['currency_legend']
    df = pd.DataFrame({k: v for k, v in flujos.items() if k != 'currency_legend'})
    df['currency'] = [leyenda[i] for i in df['currency']]
    df['settlement_date'] = [qcf.QCDate(int(s)).iso_code() for s in df['settlement_date']]
    return df

df = flows_as_dataframe(flujos)
df.style.format({
    'interest': '{0:,.2f}',
    'amortization': '{0:,.2f}',
    'total': '{0:,.2f}',
})

,op_key,leg_number,settlement_date,interest,amortization,total,currency
0,1,1,2026-07-15,"2,765,277.78",0.00,"2,765,277.78",CLP
1,1,2,2026-07-15,"-2,111.67",0.00,"-2,111.67",USD
2,2,1,2026-07-15,"6,913,194.44",0.00,"6,913,194.44",CLP
3,2,2,2026-07-15,"-5,279.17",0.00,"-5,279.17",USD
4,3,1,2026-07-15,"1,382,638.89",0.00,"1,382,638.89",CLP
5,3,2,2026-07-15,"-1,055.83",0.00,"-1,055.83",USD


## Cálculo en batch: 1.000 operaciones

El caso de uso que motiva estos objetos es el portafolio grande. Se construye un portafolio con 1.000 operaciones —la misma estructura de dos patas repetida, con claves distintas— y se mide el tiempo de las consultas. Cada operación aporta 2 patas, de modo que `states_at` devolverá 2.000 filas.

In [20]:
n_operaciones = 1_000

t0 = time.perf_counter()
portfolio_grande = qcf.Portfolio()
for key in range(1, n_operaciones + 1):
    portfolio_grande.add(build_operacion(key, 100_000_000.0, 36))
t1 = time.perf_counter()

print(f'operaciones: {portfolio_grande.size():,}')
print(f'tiempo de construcción: {t1 - t0:,.3f} segundos')

operaciones: 1,000
tiempo de construcción: 0.214 segundos


In [21]:
t0 = time.perf_counter()
estados_grande = portfolio_grande.states_at(qcf.QCDate(15, 4, 2026), curvas)
t1 = time.perf_counter()

print(f'filas (patas vivas): {len(estados_grande["op_key"]):,}')
print(f'tiempo de states_at: {(t1 - t0) * 1_000:,.1f} milisegundos')

filas (patas vivas): 2,000
tiempo de states_at: 1.9 milisegundos


Vale la pena comparar los dos tiempos: **construir** el portafolio es órdenes de magnitud más caro que **consultarlo**. Ese es justamente el diseño: las operaciones se dan de alta una vez y luego se consultan muchas veces —a distintas fechas, con distintas curvas— sin volver a recorrer la construcción de las patas.

El resultado se envuelve en un `DataFrame` de la misma forma que antes, y ya admite las agregaciones habituales.

In [22]:
df_grande = states_as_dataframe(estados_grande)
df_grande.head(6).style.format({
    'accrued_interest': '{0:,.2f}',
    'outstanding_notional': '{0:,.2f}',
    'present_value': '{0:,.2f}',
})

,op_key,leg_number,currency,accrued_interest,outstanding_notional,interest_settling,amortization_settling,total_settling,next_flow_date,present_value
0,1,1,CLP,"1,375,000.00","100,000,000.00",0.000000,0.000000,0.000000,2026-07-15,"102,489,307.36"
1,1,2,USD,"-1,050.00","-100,000.00",0.000000,0.000000,0.000000,2026-07-15,"-102,326.19"
2,2,1,CLP,"1,375,000.00","100,000,000.00",0.000000,0.000000,0.000000,2026-07-15,"102,489,307.36"
3,2,2,USD,"-1,050.00","-100,000.00",0.000000,0.000000,0.000000,2026-07-15,"-102,326.19"
4,3,1,CLP,"1,375,000.00","100,000,000.00",0.000000,0.000000,0.000000,2026-07-15,"102,489,307.36"
5,3,2,USD,"-1,050.00","-100,000.00",0.000000,0.000000,0.000000,2026-07-15,"-102,326.19"


In [23]:
resumen = df_grande.groupby('currency').agg(
    patas=('op_key', 'count'),
    devengo=('accrued_interest', 'sum'),
    capital_vigente=('outstanding_notional', 'sum'),
    valor_presente=('present_value', 'sum'),
)
resumen.style.format({
    'devengo': '{0:,.2f}',
    'capital_vigente': '{0:,.2f}',
    'valor_presente': '{0:,.2f}',
})

,patas,devengo,capital_vigente,valor_presente
currency,,,,
CLP,1000,"1,375,000,000.00","100,000,000,000.00","102,489,307,362.43"
USD,1000,"-1,050,000.00","-100,000,000.00","-102,326,190.71"


Lo mismo aplica a `flows_between`, que sobre el portafolio grande entrega todos los flujos del año.

In [24]:
t0 = time.perf_counter()
flujos_grande = portfolio_grande.flows_between(qcf.QCDate(31, 12, 2025), qcf.QCDate(31, 12, 2026))
t1 = time.perf_counter()

print(f'flujos: {len(flujos_grande["op_key"]):,}')
print(f'tiempo de flows_between: {(t1 - t0) * 1_000:,.1f} milisegundos')

flujos: 2,000
tiempo de flows_between: 2.4 milisegundos


### Paralelismo y determinismo

Ambas consultas paralelizan el trabajo entre operaciones. El argumento `num_threads` controla cuántos hilos se usan; el valor por default `0` significa *usar la concurrencia del hardware*. El resultado es **bit a bit idéntico** cualquiera sea el número de hilos: el particionamiento no altera el orden de las sumas dentro de una pata.

A esta escala la consulta ya es de pocos milisegundos, de modo que los tiempos que siguen están dominados por el costo de levantar los hilos y no muestran una ganancia interesante. La diferencia aparece con portafolios bastante más grandes.

In [25]:
tiempos = {}
for n_threads in [1, 2, 4, 8, 0]:
    t0 = time.perf_counter()
    estados_n = portfolio_grande.states_at(qcf.QCDate(15, 4, 2026), curvas, n_threads)
    t1 = time.perf_counter()
    etiqueta = 'hardware' if n_threads == 0 else str(n_threads)
    tiempos[etiqueta] = (t1 - t0) * 1_000

pd.DataFrame.from_dict(tiempos, orient='index', columns=['milisegundos']).rename_axis('num_threads')

,milisegundos
num_threads,
1,3.113029
2,2.098144
4,2.164029
8,1.578176
hardware,1.594571


In [26]:
un_hilo = portfolio_grande.states_at(qcf.QCDate(15, 4, 2026), curvas, 1)
ocho_hilos = portfolio_grande.states_at(qcf.QCDate(15, 4, 2026), curvas, 8)

for columna in ['accrued_interest', 'outstanding_notional', 'present_value', 'next_flow_date']:
    assert np.array_equal(un_hilo[columna], ocho_hilos[columna]), columna

print('resultados idénticos bit a bit con 1 y con 8 hilos')

resultados idénticos bit a bit con 1 y con 8 hilos


## `remove`

Finalmente, las operaciones se eliminan del portafolio por su clave.

In [27]:
portfolio.remove(2)
portfolio.size()

2

In [28]:
df = states_as_dataframe(portfolio.states_at(qcf.QCDate(15, 4, 2026), curvas))
df[['op_key', 'leg_number', 'currency', 'present_value']].style.format({'present_value': '{0:,.2f}'})

,op_key,leg_number,currency,present_value
0,1,1,CLP,"102,489,307.36"
1,1,2,USD,"-102,326.19"
2,3,1,CLP,"51,605,796.54"
3,3,2,USD,"-51,588.42"
